# Quick Test: Verify Model Works on Training Data

This notebook tests the trained model directly on the Phase 3 training data to verify:
1. The model is working correctly
2. The model can detect the SetMACE file in case 11
3. The issue is with prototype preprocessing, not the model itself

In [12]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("Libraries imported successfully")

Libraries imported successfully


## 1. Load Model and Data

In [13]:
print("=" * 80)

# Load the trained model
print("\n1. Loading trained model...")
model = joblib.load('/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 5 - V2 Hyperparameter Tuning/random_forest_tuned_v2.pkl')
print(f"   ✓ Model loaded: {type(model).__name__}")
print(f"   ✓ Expects {len(model.feature_names_in_)} features")

# Load Phase 3 final training data (already preprocessed)
print("\n2. Loading Phase 3 final training data...")
df = pd.read_csv('/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - V2 Feature Selection/all_cases_combined_v2_phase3_final.csv',
                 low_memory=False)
print(f"   ✓ Loaded {len(df):,} records")
print(f"   ✓ Columns: {len(df.columns)}")
print(f"   ✓ Timestomped: {(df['timestomped'] == 1).sum():,}")


1. Loading trained model...
   ✓ Model loaded: RandomForestClassifier
   ✓ Expects 51 features

2. Loading Phase 3 final training data...
   ✓ Loaded 283,118 records
   ✓ Columns: 56
   ✓ Timestomped: 280


## 2. Filter to Case 11 (Prototype Test Case)

In [14]:
print("\n" + "=" * 80)
print("CASE 11 ANALYSIS")
print("=" * 80)

# Filter to case 11
case11 = df[df['case_id'] == 11].copy()
print(f"\nCase 11 records: {len(case11):,}")
print(f"Timestomped in case 11: {(case11['timestomped'] == 1).sum()}")

# Find SetMACE rows
setmace = case11[case11['filename'].str.contains('SetMACE_SI_MACE_Copy_Manipulation', na=False)]
print(f"\nSetMACE rows found: {len(setmace)}")

if len(setmace) > 0:
    print("\nSetMACE file details:")
    for idx, row in setmace.iterrows():
        print(f"  Row {idx}:")
        print(f"    Filename: {row['filename']}")
        print(f"    Source: {row['source']}")
        print(f"    LSN: {row['lf_lsn']}")
        print(f"    Timestomped label: {row['timestomped']}")


CASE 11 ANALYSIS

Case 11 records: 0
Timestomped in case 11: 0

SetMACE rows found: 0


## 3. Prepare Features for Prediction

In [15]:
print("\n" + "=" * 80)
print("PREPARING FEATURES")
print("=" * 80)

# Define metadata columns (exclude from features)
identifier_cols = ['case_id', 'eventtime', 'eventtime_dt', 'filename', 'filepath', 'merge_key']
target_col = 'timestomped'

# Get feature columns
feature_cols = [col for col in df.columns if col not in identifier_cols + [target_col]]

print(f"\nTotal columns in data: {len(df.columns)}")
print(f"Metadata columns: {len(identifier_cols)}")
print(f"Target column: 1 ({target_col})")
print(f"Feature columns: {len(feature_cols)}")

# Extract features for case 11
X_case11 = case11[feature_cols].copy()
y_case11 = case11[target_col].copy()

print(f"\nCase 11 feature matrix BEFORE preprocessing: {X_case11.shape}")

# ==========================================
# APPLY PHASE 4/5 PREPROCESSING
# ==========================================
print(f"\nApplying Phase 4/5 preprocessing...")

# Convert boolean to int
bool_cols = X_case11.select_dtypes(include=['bool']).columns.tolist()
if bool_cols:
    print(f"  Converting {len(bool_cols)} boolean columns to int...")
    for col in bool_cols:
        X_case11[col] = X_case11[col].astype(int)

# Handle object columns
object_cols = X_case11.select_dtypes(include=['object']).columns.tolist()
print(f"  Processing {len(object_cols)} object columns...")

# Drop high-cardinality columns
cols_to_drop = [col for col in object_cols if X_case11[col].nunique() > 1000]
if cols_to_drop:
    print(f"    Dropping {len(cols_to_drop)} high-cardinality columns")
    X_case11 = X_case11.drop(columns=cols_to_drop)

# One-hot or label encode remaining object columns
cols_to_encode = [col for col in object_cols if col not in cols_to_drop and col in X_case11.columns]

from sklearn.preprocessing import LabelEncoder

for col in cols_to_encode:
    X_case11[col] = X_case11[col].fillna('missing')
    unique_count = X_case11[col].nunique()
    
    if unique_count < 5:
        # One-hot encode with drop_first=True (matching Phase 4/5)
        dummies = pd.get_dummies(X_case11[col], prefix=col, drop_first=True)
        X_case11 = pd.concat([X_case11.drop(columns=[col]), dummies], axis=1)
        print(f"    One-hot encoded: {col} ({unique_count} values) -> {len(dummies.columns)} columns")
    else:
        # Label encode
        le = LabelEncoder()
        X_case11[col] = le.fit_transform(X_case11[col])
        print(f"    Label encoded: {col} ({unique_count} values)")

# Fill missing values
X_case11 = X_case11.fillna(-1)

print(f"\n✓ Preprocessed feature matrix: {X_case11.shape}")

# Check feature alignment with model
model_features = set(model.feature_names_in_)
data_features = set(X_case11.columns)

missing = model_features - data_features
extra = data_features - model_features

print(f"\nFeature alignment check:")
print(f"  Model expects: {len(model_features)} features")
print(f"  Data has: {len(data_features)} features")
print(f"  Missing: {len(missing)} features")
print(f"  Extra: {len(extra)} features")

if missing:
    print(f"\n  Missing features:")
    for feat in sorted(list(missing))[:10]:
        print(f"    - {feat}")
    if len(missing) > 10:
        print(f"    ... and {len(missing) - 10} more")
    
    # Add missing features as 0
    for feat in missing:
        X_case11[feat] = 0
    print(f"\n  ✓ Added {len(missing)} missing features (filled with 0)")

if extra:
    print(f"\n  Extra features (will be dropped):")
    for feat in sorted(list(extra))[:10]:
        print(f"    - {feat}")
    if len(extra) > 10:
        print(f"    ... and {len(extra) - 10} more")
    
    X_case11 = X_case11.drop(columns=list(extra))
    print(f"\n  ✓ Dropped {len(extra)} extra features")

# Align column order
X_case11_aligned = X_case11[model.feature_names_in_]
print(f"\n✓ Final aligned feature matrix: {X_case11_aligned.shape}")


PREPARING FEATURES

Total columns in data: 56
Metadata columns: 6
Target column: 1 (timestomped)
Feature columns: 49

Case 11 feature matrix BEFORE preprocessing: (0, 49)

Applying Phase 4/5 preprocessing...
  Converting 12 boolean columns to int...
  Processing 19 object columns...
    One-hot encoded: lf_event (0 values) -> 0 columns
    One-hot encoded: lf_target_vcn (0 values) -> 0 columns
    One-hot encoded: usn_event_info (0 values) -> 0 columns
    One-hot encoded: usn_file_reference_number (0 values) -> 0 columns
    One-hot encoded: usn_parent_file_reference_number (0 values) -> 0 columns
    One-hot encoded: source (0 values) -> 0 columns
    One-hot encoded: lf_creation_time_before (0 values) -> 0 columns
    One-hot encoded: lf_creation_time_after (0 values) -> 0 columns
    One-hot encoded: lf_modified_time_before (0 values) -> 0 columns
    One-hot encoded: lf_modified_time_after (0 values) -> 0 columns
    One-hot encoded: lf_accessed_time_before (0 values) -> 0 column

## 4. Run Predictions on Case 11

In [16]:
print("\n" + "=" * 80)
print("PREPARING FEATURES")
print("=" * 80)

# Define metadata columns (exclude from features)
identifier_cols = ['case_id', 'eventtime', 'eventtime_dt', 'filename', 'filepath', 'merge_key']
target_col = 'timestomped'

# Get feature columns
feature_cols = [col for col in df.columns if col not in identifier_cols + [target_col]]

print(f"\nTotal columns in data: {len(df.columns)}")
print(f"Metadata columns: {len(identifier_cols)}")
print(f"Target column: 1 ({target_col})")
print(f"Feature columns: {len(feature_cols)}")

# Extract features for case 11
X_case11 = case11[feature_cols].copy()
y_case11 = case11[target_col].copy()

print(f"\nCase 11 records: {len(X_case11)}")
print(f"Case 11 feature matrix BEFORE preprocessing: {X_case11.shape}")

# ==========================================
# APPLY PHASE 4/5 PREPROCESSING
# ==========================================
print(f"\nApplying Phase 4/5 preprocessing...")

# Convert boolean to int
bool_cols = X_case11.select_dtypes(include=['bool']).columns.tolist()
if bool_cols:
    print(f"  Converting {len(bool_cols)} boolean columns to int...")
    for col in bool_cols:
        X_case11[col] = X_case11[col].astype(int)

print(f"  After boolean conversion: {X_case11.shape}")

# Handle object columns
object_cols = X_case11.select_dtypes(include=['object']).columns.tolist()
print(f"  Processing {len(object_cols)} object columns...")

# Drop high-cardinality columns
cols_to_drop = [col for col in object_cols if X_case11[col].nunique() > 1000]
if cols_to_drop:
    print(f"    Dropping {len(cols_to_drop)} high-cardinality columns")
    X_case11 = X_case11.drop(columns=cols_to_drop)

print(f"  After dropping high-cardinality: {X_case11.shape}")

# One-hot or label encode remaining object columns
cols_to_encode = [col for col in object_cols if col not in cols_to_drop and col in X_case11.columns]

from sklearn.preprocessing import LabelEncoder

for col in cols_to_encode:
    X_case11[col] = X_case11[col].fillna('missing')
    unique_count = X_case11[col].nunique()
    
    if unique_count < 5:
        # One-hot encode with drop_first=True (matching Phase 4/5)
        dummies = pd.get_dummies(X_case11[col], prefix=col, drop_first=True)
        X_case11 = pd.concat([X_case11.drop(columns=[col]), dummies], axis=1)
        print(f"    One-hot encoded: {col} ({unique_count} values) -> {len(dummies.columns)} columns")
    else:
        # Label encode
        le = LabelEncoder()
        X_case11[col] = le.fit_transform(X_case11[col])
        print(f"    Label encoded: {col} ({unique_count} values)")

print(f"  After encoding: {X_case11.shape}")

# Fill missing values
X_case11 = X_case11.fillna(-1)

print(f"\n✓ Preprocessed feature matrix: {X_case11.shape}")
print(f"  Rows: {len(X_case11)}")
print(f"  Columns: {len(X_case11.columns)}")

# Check feature alignment with model
model_features = set(model.feature_names_in_)
data_features = set(X_case11.columns)

missing = model_features - data_features
extra = data_features - model_features

print(f"\nFeature alignment check:")
print(f"  Model expects: {len(model_features)} features")
print(f"  Data has: {len(data_features)} features")
print(f"  Missing: {len(missing)} features")
print(f"  Extra: {len(extra)} features")

if missing:
    print(f"\n  Missing features:")
    for feat in sorted(list(missing))[:10]:
        print(f"    - {feat}")
    if len(missing) > 10:
        print(f"    ... and {len(missing) - 10} more")
    
    # Add missing features as 0
    for feat in missing:
        X_case11[feat] = 0
    print(f"\n  ✓ Added {len(missing)} missing features (filled with 0)")

print(f"  After adding missing: {X_case11.shape}")

if extra:
    print(f"\n  Extra features (will be dropped):")
    for feat in sorted(list(extra))[:10]:
        print(f"    - {feat}")
    if len(extra) > 10:
        print(f"    ... and {len(extra) - 10} more")
    
    X_case11 = X_case11.drop(columns=list(extra))
    print(f"\n  ✓ Dropped {len(extra)} extra features")

print(f"  After dropping extra: {X_case11.shape}")

# Align column order
X_case11_aligned = X_case11[model.feature_names_in_]
print(f"\n✓ Final aligned feature matrix: {X_case11_aligned.shape}")
print(f"  Final rows: {len(X_case11_aligned)}")
print(f"  Final columns: {len(X_case11_aligned.columns)}")

if len(X_case11_aligned) == 0:
    print("\n❌ ERROR: Feature matrix is empty!")
    print("   This shouldn't happen - investigating...")


PREPARING FEATURES

Total columns in data: 56
Metadata columns: 6
Target column: 1 (timestomped)
Feature columns: 49

Case 11 records: 0
Case 11 feature matrix BEFORE preprocessing: (0, 49)

Applying Phase 4/5 preprocessing...
  Converting 12 boolean columns to int...
  After boolean conversion: (0, 49)
  Processing 19 object columns...
  After dropping high-cardinality: (0, 49)
    One-hot encoded: lf_event (0 values) -> 0 columns
    One-hot encoded: lf_target_vcn (0 values) -> 0 columns
    One-hot encoded: usn_event_info (0 values) -> 0 columns
    One-hot encoded: usn_file_reference_number (0 values) -> 0 columns
    One-hot encoded: usn_parent_file_reference_number (0 values) -> 0 columns
    One-hot encoded: source (0 values) -> 0 columns
    One-hot encoded: lf_creation_time_before (0 values) -> 0 columns
    One-hot encoded: lf_creation_time_after (0 values) -> 0 columns
    One-hot encoded: lf_modified_time_before (0 values) -> 0 columns
    One-hot encoded: lf_modified_time

## 5. Evaluate Results

In [17]:
print("\n" + "=" * 80)
print("RESULTS")
print("=" * 80)

# Overall metrics for case 11
print(f"\nCase 11 Performance:")
print(f"  Accuracy:  {accuracy_score(y_case11, predictions):.2%}")

if (y_case11 == 1).sum() > 0:
    print(f"  Precision: {precision_score(y_case11, predictions, zero_division=0):.2%}")
    print(f"  Recall:    {recall_score(y_case11, predictions, zero_division=0):.2%}")
    print(f"  F1-Score:  {f1_score(y_case11, predictions, zero_division=0):.4f}")
    
    # Confusion matrix
    cm = confusion_matrix(y_case11, predictions)
    tn, fp, fn, tp = cm.ravel()
    print(f"\n  Confusion Matrix:")
    print(f"    TN: {tn:4d}  FP: {fp:4d}")
    print(f"    FN: {fn:4d}  TP: {tp:4d}")

# SetMACE specific results
print(f"\n" + "=" * 80)
print("SETMACE FILE RESULTS")
print("=" * 80)

if len(setmace) > 0:
    setmace_results = case11[case11['filename'].str.contains('SetMACE_SI_MACE_Copy_Manipulation', na=False)]
    
    for idx, row in setmace_results.iterrows():
        print(f"\nRow {idx}:")
        print(f"  Filename: {row['filename']}")
        print(f"  Source: {row['source']}")
        print(f"  Actual label: {row['timestomped']} ({'Timestomped' if row['timestomped'] == 1 else 'Benign'})")
        print(f"  Predicted: {row['predicted']} ({'Timestomped' if row['predicted'] == 1 else 'Benign'})")
        print(f"  Confidence: {row['confidence']:.2%}")
        
        if row['predicted'] == row['timestomped']:
            print(f"  Result: ✓ CORRECT")
        else:
            print(f"  Result: ✗ INCORRECT")
else:
    print("\n⚠️  No SetMACE file found!")


RESULTS

Case 11 Performance:


NameError: name 'predictions' is not defined

## 6. Show Top Confident Predictions

In [ ]:
print("\n" + "=" * 80)
print("TOP PREDICTIONS")
print("=" * 80)

# Top 10 highest confidence timestomped predictions
print(f"\nTop 10 Highest Confidence Timestomped Predictions:")
top10 = case11.nlargest(10, 'confidence')[['filename', 'source', 'timestomped', 'predicted', 'confidence']]
print(top10.to_string(index=True))

## 7. Conclusion

In [ ]:
print("\n" + "=" * 80)
print("CONCLUSION")
print("=" * 80)

if len(setmace) > 0:
    setmace_results = case11[case11['filename'].str.contains('SetMACE_SI_MACE_Copy_Manipulation', na=False)]
    
    if all(setmace_results['predicted'] == setmace_results['timestomped']):
        print("\n✓ SUCCESS: Model correctly predicts SetMACE file as timestomped!")
        print("  - The model IS working correctly on the training data")
        print("  - The issue is with the prototype preprocessing, not the model")
        print("  - Next step: Fix prototype notebooks to match training preprocessing")
    else:
        print("\n✗ FAILURE: Model incorrectly predicts SetMACE file!")
        print("  - This suggests a deeper model training issue")
        print("  - The model may not have learned the timestomping patterns correctly")
        print("  - Need to investigate feature engineering and model training")
else:
    print("\n⚠️  WARNING: No SetMACE file found in case 11")
    print("  - This is unexpected - the SetMACE file should exist")
    print("  - Need to verify data loading")

print("\n" + "=" * 80)